## AI4Climate ML tutorial - Evaluation
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In previous notebooks we have trained a few ML models to predict the climate zone of a land point on earth from the climate means for that point. In this notebook we will dig deeper into how well our trained models are performing and consider different ways to measure that.


### Prerequisites 
- Same as previous notebooks
- Have completed model training pipeline notebook


### Learning outcomes from completing the notebook
- Load model and data and make predictions
- Calculate a range of metrics for a classification problem
- Break down the results by target class and other data subsets to gain insight into when the model performs better or worse.


## Tutorial - Model Evaluation
To determine whether the algorithm is performing well, we need to measure the performance in some way. For supervised learning, metrics are divided by the two key types of problem: regression and classification. Whichever sort of algorithm you're using, getting good results depends on measuring the right thing that best reflects the real world impact, business need or physical reality of the system being studied. Choosing the wrong metric might result in a trained algorithm that seemingly performs well in development, but isn't actually that useful in the real world.

We're dealing with a classification problem here, so we will use classfication relevant metrics.
The obvious metric to use is accuracy i.e. how many predictions did we get right? As we will see, this  usually does represent the way want the algorithm to perform. Obviously we want all the predictions to be correct, but generally, some will be wrong. Depending on the impact of incorrect predictions, we may wish penalise certain mistakes more or less. Different sorts of metrics allow us to do that.
    
A common starting point is precision and recall. This is frequently confusing, and the [wikipedia page](https://en.wikipedia.org/wiki/Precision_and_recall) has numerous useful explanations and diagrams. 


* **Precision** is the portion of data points predicted as being in a class that are actually in that class.
  * In other words, `Precision = P( true positive / classified as positive)`
* **Recall** in the proportion of data points that are in a specific class in reality that are predicted as being in that class.
   * In other words, `Recall = P( correctly classified / total observed positive)`

In the diagram below the events in the circle are all the predicted events (eg, predicted rotors to happen). The left side of the square are the total number of actual events that occurred. 

![Precision and recall](https://raw.githubusercontent.com/MetOffice/ml_weather_tutorial/refs/heads/main/images/Precisionrecall.svg)

Graphic taken from Wikipedia

### Import libraries

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import pandas

In [3]:
import sklearn
import sklearn.preprocessing
import sklearn.tree
import torch

In [4]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'mo_linux',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, v

In [5]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [6]:
current_platform = tutorial_config['platform']

In [7]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones')

In [8]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready')

In [9]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [10]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

### Load data for inference

In [11]:
current_res = 1.0

In [12]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/data/users/dscop/ml_tutorial/climate_zones/ml_ready/climate_zones_1p0.csv')

In [ ]:
zones_df = pandas.read_csv(mlready_data_path)

In [ ]:
# reducing the total data point to decrease memeory requirements
# zones_df = zones_df[zones_df['scenario'] == 'historic']
zones_df = zones_df[(zones_df['period_start']==1991)&(zones_df['scenario']=='historic')]

In [ ]:
zones_df

In [ ]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [ ]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

In [ ]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


By setting a random seed, we are ensuring that we get the train/val/test split in each of the notebooks.

In [ ]:
random_seed = tutorial_config['random_seed']

In [ ]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [ ]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [ ]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


In [ ]:
input_scaler = sklearn.preprocessing.StandardScaler()
input_scaler.fit(train_df[predictors])


In [ ]:
input_scaler.mean_

In [ ]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [ ]:
train_df[[target_var]].value_counts()

In [ ]:
target_encoder = sklearn.preprocessing.LabelEncoder()
target_encoder.fit(train_df[[target_var]])

In [ ]:
y_train = target_encoder.transform(train_df[[target_var]])
y_val = target_encoder.transform(val_df[[target_var]])
y_test = target_encoder.transform(test_df[[target_var]])


In [ ]:
dt_opts = {'max_depth':10, 'min_samples_leaf': 2, 'min_samples_split': 5}

In [ ]:
dt_clf = sklearn.tree.DecisionTreeClassifier(**dt_opts)

In [ ]:
%%time
dt_clf.fit(X_train, y_train) 

In [ ]:
dt_clf

In [ ]:
y_pred_train = dt_clf.predict(X_train)
y_pred_val = dt_clf.predict(X_val)

In [ ]:
# visualise the dsitribution of predictions
# compare for different periods and scenarios


In [ ]:
# error analysis - look at results for different time periods and scenarios

In [ ]:
# display predictions on a map

In [ ]:
sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_train)

In [ ]:
sklearn.metrics.precision_recall_fscore_support(y_val, y_pred_val)

In [ ]:
target_encoder.inverse_transform(y_pred_train)

In [ ]:
sklearn.metrics.confusion_matrix(target_encoder.inverse_transform(y_train), target_encoder.inverse_transform(y_pred_train))

In [ ]:
sklearn.metrics.confusion_matrix(target_encoder.inverse_transform(y_train), target_encoder.inverse_transform(y_pred_train))

## Exercises
for students to try that do not have solutions but maybe have an answer or benchmark to facilitate understanding


### Next steps or potential follow on material



###  Exmaples of Use


### Data statement
###     References
